# Vesuvius V7-Fixed: Practical Topology-Aware Training

## Fixes from V7:
1. **Pre-compute SDF/Skeleton** during data loading (not in __getitem__)
2. **Lighter skeleton decoder** - share more with main decoder
3. **Simpler attention** - scSE works well, D-LKA adds complexity
4. **Skeleton loss stability** - use soft skeleton, not GT skeleton

## Key Changes:
- Skeleton from epoch 0 (not epoch 300)
- Reduced Dice weight (0.2 vs 0.3)
- SDF auxiliary target
- Single decoder with multi-head output

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# =============================================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy.ndimage import distance_transform_edt

try:
    import tifffile
    USE_TIFFFILE = True
except ImportError:
    USE_TIFFFILE = False

warnings.filterwarnings('ignore')

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v7")
    
    # Model
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None
    N_BLOCKS: List[int] = None
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    USE_SDF_HEAD: bool = True  # Multi-head output
    
    # Training
    EPOCHS: int = 1200
    BATCH_SIZE: int = 4
    NUM_WORKERS: int = 16
    PREFETCH_FACTOR: int = 4
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # V7-Fixed Loss Weights (topology-focused)
    DICE_WEIGHT: float = 0.2       # Reduced
    BCE_WEIGHT: float = 0.1        # Reduced
    SKELETON_WEIGHT: float = 0.3   # Increased - key for topology
    CLDICE_WEIGHT: float = 0.3     # Increased
    SDF_WEIGHT: float = 0.1        # Auxiliary
    
    # Loss scheduling - START SKELETON EARLY!
    SKELETON_START_EPOCH: int = 0    # From epoch 0!
    CLDICE_START_EPOCH: int = 200    # Earlier than before
    
    # Training control
    RESUME_TRAINING: bool = True
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 10
    
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 8
    PRELOAD_ALL: bool = True
    PRECOMPUTE_SDF: bool = True  # Pre-compute during loading!
    
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = Config()
cfg.__post_init__()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)

print("="*60)
print("V7-FIXED: PRACTICAL TOPOLOGY-AWARE TRAINING")
print("="*60)
print(f"Key changes from V7:")
print(f"  - Pre-computed SDF (no slowdown)")
print(f"  - Single decoder with multi-head")
print(f"  - Skeleton from epoch 0")
print(f"  - Reduced Dice weight: {cfg.DICE_WEIGHT}")
print("="*60)

In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER (same as before)
# =============================================================================

class CheckpointManager:
    def __init__(self, save_dir: Path, load_dir: Path = None, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
        }
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best! Val Dice: {val_dice:.4f}")
        
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
        
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None, load_best=False) -> int:
        path = self.load_dir / ('best_model.pth' if load_best else 'last_checkpoint.pth')
        if not path.exists():
            print("No checkpoint found, starting fresh")
            return 0
        
        ckpt = torch.load(path, map_location=cfg.DEVICE, weights_only=False)
        model_to_load = model._orig_mod if hasattr(model, '_orig_mod') else model
        model_to_load.load_state_dict(ckpt['model_state_dict'])
        
        if optimizer: optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt.get('scheduler_state_dict'): scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'): scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        print(f"Resumed from epoch {ckpt['epoch']}, best: {self.best_score:.4f}")
        return ckpt['epoch'] + 1

print("CheckpointManager ready")

In [ ]:
# =============================================================================
# CELL 3: DATASET WITH PRE-COMPUTED SDF (FAST!)
# =============================================================================

def load_volume(path: Path) -> np.ndarray:
    if USE_TIFFFILE:
        return tifffile.imread(str(path))
    else:
        im = Image.open(path)
        return np.stack([np.array(p) for p in ImageSequence.Iterator(im)], axis=0)


def compute_sdf_full(mask: np.ndarray) -> np.ndarray:
    """Compute SDF for FULL volume (done once during loading)."""
    dist_fg = distance_transform_edt(mask > 0)
    dist_bg = distance_transform_edt(mask == 0)
    sdf = dist_fg - dist_bg
    sdf = np.clip(sdf / 10.0, -1, 1)  # Normalize to [-1, 1]
    return sdf.astype(np.float32)


class VesuviusDatasetV7Fixed(Dataset):
    """
    V7-Fixed Dataset:
    - Pre-computes SDF during loading (NOT in __getitem__)
    - No skeleton GT (use soft skeleton in loss)
    - Fast augmentation only
    """
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        preload: bool = True,
        precompute_sdf: bool = True,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.precompute_sdf = precompute_sdf
        
        df = pd.read_csv(csv_path)
        valid_ids = [idx for idx in df['id'].values 
                     if (self.images_dir / f"{idx}.tif").exists() and 
                        (self.labels_dir / f"{idx}.tif").exists()]
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.volumes = {}
        self.sdfs = {}  # Pre-computed SDFs
        self.fg_coords = {}
        
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                img = (img - img.mean()) / (img.std() + 1e-8)
                self.volumes[vol_id] = (img, lbl)
                
                # Pre-compute SDF for full volume!
                if precompute_sdf:
                    fg_mask = (lbl == 1)
                    self.sdfs[vol_id] = compute_sdf_full(fg_mask)
                
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            print(f"Loaded {len(self.volumes)} volumes with pre-computed SDF")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        sdf_full = self.sdfs.get(vol_id)
        
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d, pad_h, pad_w = max(0, pd-d), max(0, ph-h), max(0, pw-w)
            image = np.pad(image, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
            label = np.pad(label, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='constant', constant_values=2)
            if sdf_full is not None:
                sdf_full = np.pad(sdf_full, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
            d, h, w = image.shape
        
        # Foreground-centered sampling (70%)
        fg = self.fg_coords.get(vol_id)
        if self.is_train and random.random() < 0.7 and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z = max(0, min(c[0] - pd//2, d - pd))
            y = max(0, min(c[1] - ph//2, h - ph))
            x = max(0, min(c[2] - pw//2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        sdf_patch = sdf_full[z:z+pd, y:y+ph, x:x+pw].copy() if sdf_full is not None else None
        
        # Fast augmentation
        if self.is_train:
            for ax in range(3):
                if random.random() > 0.5:
                    img_patch = np.flip(img_patch, ax)
                    lbl_patch = np.flip(lbl_patch, ax)
                    if sdf_patch is not None:
                        sdf_patch = np.flip(sdf_patch, ax)
            
            k = random.randint(0, 3)
            if k:
                img_patch = np.rot90(img_patch, k, (1,2))
                lbl_patch = np.rot90(lbl_patch, k, (1,2))
                if sdf_patch is not None:
                    sdf_patch = np.rot90(sdf_patch, k, (1,2))
            
            img_patch = np.ascontiguousarray(img_patch)
            lbl_patch = np.ascontiguousarray(lbl_patch)
            if sdf_patch is not None:
                sdf_patch = np.ascontiguousarray(sdf_patch)
            
            if random.random() > 0.5:
                img_patch = img_patch * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_patch = img_patch + random.uniform(-0.1, 0.1)
        
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        result = {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }
        
        if sdf_patch is not None:
            result['sdf'] = torch.from_numpy(sdf_patch).unsqueeze(0).float()
        
        return result

print("V7-Fixed Dataset ready (pre-computed SDF = no slowdown!)")

In [ ]:
# =============================================================================
# CELL 4: MODEL - SINGLE DECODER WITH MULTI-HEAD OUTPUT
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        self.norm = nn.InstanceNorm3d(num_features, eps=eps, affine=False)
    
    def forward(self, x):
        if torch.isnan(x).any() or torch.isinf(x).any():
            x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        if x.shape[2] < 2 or x.shape[3] < 2 or x.shape[4] < 2:
            mean = x.mean(dim=(2,3,4), keepdim=True)
            var = x.var(dim=(2,3,4), keepdim=True, unbiased=False)
            x = (x - mean) / torch.sqrt(var + self.eps)
        else:
            x = self.norm(x)
        if self.affine:
            x = x * self.weight.view(1,-1,1,1,1) + self.bias.view(1,-1,1,1,1)
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x): return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    def forward(self, x): return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b,c))))).view(b,c,1,1,1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class MultiHeadUNet3D(nn.Module):
    """
    V7-Fixed: Single decoder with multiple output heads.
    More efficient than dual-stream, same expressivity.
    
    Heads:
    - seg: Main segmentation
    - sdf: Signed distance function (topology-aware)
    """
    
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
        use_sdf_head: bool = True,
    ):
        super().__init__()
        
        if features is None: features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None: n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.use_sdf_head = use_sdf_head
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            self.encoders.append(nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            ))
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], kernel_size=2, stride=2))
            self.dec_convs.append(ConvBlock(features[i] * 2, features[i]))
        
        # Output heads
        self.seg_head = nn.Conv3d(features[0], out_ch, 1)
        if use_sdf_head:
            self.sdf_head = nn.Conv3d(features[0], out_ch, 1)
        
        # Deep supervision
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) for i in range(min(3, len(features)-1))
            ])
    
    def forward(self, x):
        # Encoder
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        # Decoder
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        # Outputs
        seg_out = self.seg_head(x)
        
        if self.training:
            result = {'seg': seg_out, 'deep': ds_outputs}
            if self.use_sdf_head:
                result['sdf'] = self.sdf_head(x)
            return result
        return seg_out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("MultiHeadUNet3D ready")
print("  - Single decoder (efficient)")
print("  - Multiple heads: seg + sdf")
print("  - scSE attention")

In [ ]:
# =============================================================================
# CELL 5: TOPOLOGY-FOCUSED LOSS (SKELETON FROM EPOCH 0)
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        inter = (pred * target).sum()
        union = pred.sum() + target.sum()
        return 1 - (2 * inter + self.smooth) / (union + self.smooth)


class BCEWithMask(nn.Module):
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            return (loss * valid).sum() / (valid.sum() + 1e-8)
        return F.binary_cross_entropy_with_logits(pred, target)


def soft_skeletonize(x, num_iter=10):
    """Differentiable soft skeletonization."""
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SkeletonRecallLoss(nn.Module):
    """Ensures model predicts the skeleton correctly."""
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        
        # Use SOFT skeleton of target (differentiable)
        target_skel = soft_skeletonize(target, num_iter=10)
        # Dilate skeleton slightly for tolerance
        target_tube = F.max_pool3d(target_skel, 5, stride=1, padding=2)
        
        recall = (pred_sig * target_tube).sum() / (target_tube.sum() + self.smooth)
        return 1 - recall


class SoftClDiceLoss(nn.Module):
    """Centerline Dice - preserves topology."""
    def __init__(self, smooth=1e-5, num_iter=10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        return 1 - cldice


class SDFLoss(nn.Module):
    """SDF regression loss."""
    def forward(self, pred_sdf, target_sdf, mask=None):
        pred_sdf = torch.tanh(pred_sdf)
        if mask is not None:
            valid = 1 - mask
            loss = F.mse_loss(pred_sdf, target_sdf, reduction='none')
            return (loss * valid).sum() / (valid.sum() + 1e-8)
        return F.mse_loss(pred_sdf, target_sdf)


class V7FixedLoss(nn.Module):
    """
    V7-Fixed Loss:
    - Skeleton from epoch 0 (not 300)
    - Reduced Dice weight
    - SDF auxiliary
    """
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.bce = BCEWithMask()
        self.skeleton = SkeletonRecallLoss()
        self.cldice = SoftClDiceLoss()
        self.sdf = SDFLoss()
        self.ds_weights = [0.5, 0.25, 0.125]
    
    def _scale(self, epoch, start, warmup=200):
        if epoch < start: return 0.0
        if epoch < start + warmup: return (epoch - start) / warmup
        return 1.0
    
    def forward(self, output, targets, epoch):
        seg = output['seg']
        sdf_pred = output.get('sdf')
        deep = output.get('deep', [])
        
        target = targets['label']
        ignore = targets['ignore_mask']
        sdf_target = targets.get('sdf')
        
        # Scales
        skel_scale = self._scale(epoch, cfg.SKELETON_START_EPOCH)  # From epoch 0!
        cldice_scale = self._scale(epoch, cfg.CLDICE_START_EPOCH)
        
        # Losses
        dice_loss = self.dice(seg, target, ignore)
        bce_loss = self.bce(seg, target, ignore)
        skel_loss = self.skeleton(seg, target, ignore) if skel_scale > 0 else torch.tensor(0.0, device=seg.device)
        cl_loss = self.cldice(seg, target, ignore) if cldice_scale > 0 else torch.tensor(0.0, device=seg.device)
        
        total = (
            cfg.DICE_WEIGHT * dice_loss +
            cfg.BCE_WEIGHT * bce_loss +
            cfg.SKELETON_WEIGHT * skel_scale * skel_loss +
            cfg.CLDICE_WEIGHT * cldice_scale * cl_loss
        )
        
        # SDF loss
        sdf_loss_val = 0.0
        if sdf_pred is not None and sdf_target is not None:
            sdf_loss = self.sdf(sdf_pred, sdf_target, ignore)
            total = total + cfg.SDF_WEIGHT * sdf_loss
            sdf_loss_val = sdf_loss.item()
        
        # Deep supervision
        for i, ds in enumerate(deep):
            if i >= len(self.ds_weights): break
            ds_t = F.interpolate(target, size=ds.shape[2:], mode='nearest')
            ds_m = F.interpolate(ignore, size=ds.shape[2:], mode='nearest')
            total = total + self.ds_weights[i] * self.dice(ds, ds_t, ds_m)
        
        return {
            'total': total,
            'dice': dice_loss.item(),
            'bce': bce_loss.item(),
            'skeleton': skel_loss.item() if skel_scale > 0 else 0.0,
            'cldice': cl_loss.item() if cldice_scale > 0 else 0.0,
            'sdf': sdf_loss_val,
        }

print("V7-Fixed Loss ready")
print(f"  Skeleton starts: epoch {cfg.SKELETON_START_EPOCH}")
print(f"  clDice starts: epoch {cfg.CLDICE_START_EPOCH}")
print(f"  Weights: Dice={cfg.DICE_WEIGHT}, Skel={cfg.SKELETON_WEIGHT}, clDice={cfg.CLDICE_WEIGHT}")

In [ ]:
# =============================================================================
# CELL 6: TRAINING LOOP
# =============================================================================

import sys
import time

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total_loss, total_dice, n = 0, 0, 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}", file=sys.stdout)
    for batch in pbar:
        images = batch['image'].to(device, non_blocking=True)
        targets = {
            'label': batch['label'].to(device, non_blocking=True),
            'ignore_mask': batch['ignore_mask'].to(device, non_blocking=True),
        }
        if 'sdf' in batch:
            targets['sdf'] = batch['sdf'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, targets, epoch)
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        n += 1
        pbar.set_postfix({'loss': f"{losses['total'].item():.3f}", 'dice': f"{losses['dice']:.3f}", 'skel': f"{losses['skeleton']:.3f}"})
    
    return {'train_loss': total_loss/n, 'train_dice': total_dice/n}


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_dice, n = 0, 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            probs = torch.sigmoid(model(images)).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b,0] > 0.5).astype(np.uint8)
            tgt = labels[b,0].astype(np.uint8)
            ign = ignore[b,0] > 0.5
            pred[ign], tgt[ign] = 0, 0
            inter = (pred & tgt).sum()
            total_dice += (2*inter + 1e-5) / (pred.sum() + tgt.sum() + 1e-5)
            n += 1
    
    return {'val_dice': total_dice / max(n, 1)}


def main_training():
    print("="*60)
    print("V7-FIXED TRAINING")
    print("="*60)
    
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    print("\n[1/4] Loading data with pre-computed SDF...")
    train_ds = VesuviusDatasetV7Fixed(
        train_csv, train_images, train_labels,
        cfg.PATCH_SIZE, is_train=True, data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME, precompute_sdf=cfg.PRECOMPUTE_SDF
    )
    val_ds = VesuviusDatasetV7Fixed(
        train_csv, train_images, train_labels,
        cfg.PATCH_SIZE, is_train=False, data_fraction=0.15,
        patches_per_volume=1, precompute_sdf=cfg.PRECOMPUTE_SDF
    )
    
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                               num_workers=cfg.NUM_WORKERS, pin_memory=True,
                               drop_last=True, persistent_workers=True, prefetch_factor=cfg.PREFETCH_FACTOR)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, num_workers=4, pin_memory=True)
    
    print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")
    
    print("\n[2/4] Creating model...")
    model = MultiHeadUNet3D(
        features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE, use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
        use_sdf_head=cfg.USE_SDF_HEAD
    ).to(cfg.DEVICE)
    
    if hasattr(torch, 'compile'):
        model = torch.compile(model, mode='reduce-overhead')
    print(f"Parameters: {count_parameters(model):,}")
    
    criterion = V7FixedLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.LEARNING_RATE * cfg.BATCH_SIZE,
                                 momentum=cfg.MOMENTUM, weight_decay=cfg.WEIGHT_DECAY, nesterov=True)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda e: (1 - e/cfg.EPOCHS)**0.9)
    scaler = GradScaler(enabled=cfg.USE_AMP)
    ckpt = CheckpointManager(cfg.CHECKPOINT_DIR)
    
    print("\n[3/4] Checking checkpoint...")
    start_epoch = ckpt.load(model, optimizer, scheduler, scaler) if cfg.RESUME_TRAINING else 0
    
    print(f"\n[4/4] Training from epoch {start_epoch+1}...")
    print("="*60)
    
    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        train_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch)
        scheduler.step()
        val_m = validate(model, val_loader, cfg.DEVICE) if (epoch+1) % cfg.VALIDATE_EVERY == 0 else {'val_dice': 0}
        
        print(f"Epoch {epoch+1}/{cfg.EPOCHS} | {time.time()-t0:.1f}s | "
              f"Loss: {train_m['train_loss']:.4f} | Dice: {train_m['train_dice']:.4f}" +
              (f" | Val: {val_m['val_dice']:.4f}" if val_m['val_dice'] > 0 else ""))
        ckpt.save(model, optimizer, scheduler, scaler, epoch, {**train_m, **val_m})
    
    print(f"\nDone! Best: {ckpt.best_score:.4f} @ epoch {ckpt.best_epoch}")
    return model, ckpt

print("Training ready!")

In [ ]:
# =============================================================================
# CELL 7: RUN TRAINING
# =============================================================================

cfg.RESUME_TRAINING = True
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 1200

# UNCOMMENT TO START:
# model, ckpt = main_training()